<a href="https://colab.research.google.com/github/xfalconix/rag-internal-policies/blob/main/RAG_minimo_politicas_internas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#EJERCICIO RAG

#PRIMER BLOQUE, DOCUMENTACIÓN:

documentos = [
    {
        "id": "ES-VAC-01",
        "pais": "España",
        "nivel_acceso": "empleado",
        "texto": "Los empleados disponen de 22 días laborables de vacaciones."
    },
    {
        "id": "ES-REM-01",
        "pais": "España",
        "nivel_acceso": "rrhh",
        "texto": "Las bandas salariales solo pueden ser consultadas por Recursos Humanos."
    },
    {
        "id": "FR-VAC-01",
        "pais": "Francia",
        "nivel_acceso": "empleado",
        "texto": "Los empleados disponen de 25 días laborables de vacaciones."
    },
    {
        "id": "ES-FOR-01",
        "pais": "España",
        "nivel_acceso": "empleado",
        "texto": "Cada empleado dispone de 20 horas anuales de formación."
    }
]

usuario = {
    "pais": "España",
    "nivel_acceso": "empleado"
}

for documento in documentos:
  print(documento["id"],documento["texto"])




ES-VAC-01 Los empleados disponen de 22 días laborables de vacaciones.
ES-REM-01 Las bandas salariales solo pueden ser consultadas por Recursos Humanos.
FR-VAC-01 Los empleados disponen de 25 días laborables de vacaciones.
ES-FOR-01 Cada empleado dispone de 20 horas anuales de formación.


In [ ]:
#SEGUNDO BLOQUE: FILTRO DE SEGURIDAD

documentos_permitidos = []

for documento in documentos:
    if (
        documento["pais"] == usuario["pais"]
        and documento["nivel_acceso"] == usuario["nivel_acceso"]
    ):
        documentos_permitidos.append(documento)

print("Documentos permitidos:")

for documento in documentos_permitidos:
    print(documento["id"], documento["texto"])

Documentos permitidos:
ES-VAC-01 Los empleados disponen de 22 días laborables de vacaciones.
ES-FOR-01 Cada empleado dispone de 20 horas anuales de formación.


In [ ]:
#Tercer bloque: crear embeddings
#USAMOS : https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

# Instalamos la librería en Google Colab
!pip install -q sentence-transformers

# Importamos la clase que permite cargar el modelo de embeddings
from sentence_transformers import SentenceTransformer

# Extraemos únicamente los textos autorizados
textos_permitidos = []

for documento in documentos_permitidos:
    textos_permitidos.append(documento["texto"])

# Cargamos un modelo de embeddings multilingüe
modelo_embeddings = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Convertimos cada texto en un vector numérico
embeddings_documentos = modelo_embeddings.encode(textos_permitidos)


print("Los documentos permitodos del bloque anterior (diccionarios):")
print(documentos_permitidos)
print()
print("Se cargan a Textos permitidos(solo la parte del texto):")
print(textos_permitidos)
print()
print("Tipo de embeddings:")
print(type(embeddings_documentos))
print()
print("Dimensiones:")
print(embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Los documentos permitodos del bloque anterior (diccionarios):
[{'id': 'ES-VAC-01', 'pais': 'España', 'nivel_acceso': 'empleado', 'texto': 'Los empleados disponen de 22 días laborables de vacaciones.'}, {'id': 'ES-FOR-01', 'pais': 'España', 'nivel_acceso': 'empleado', 'texto': 'Cada empleado dispone de 20 horas anuales de formación.'}]

Se cargan a Textos permitidos(solo la parte del texto):
['Los empleados disponen de 22 días laborables de vacaciones.', 'Cada empleado dispone de 20 horas anuales de formación.']

Tipo de embeddings:
<class 'numpy.ndarray'>

Dimensiones:
(2, 384)


In [ ]:
#Dimensiones:(2,384) significa tenemos dos documentos autorizados, c/u representado mediante vector de 384 numeros.


In [ ]:
#Cuarto bloque: representar la pregunta y comparar

from sklearn.metrics.pairwise import cosine_similarity

pregunta = "¿Tengo derecho a horas de formación?"

# Los corchetes hacen que enviemos una lista con una pregunta
embedding_pregunta = modelo_embeddings.encode([pregunta])

# Comparamos la pregunta con todos los documentos autorizados
similitudes = cosine_similarity(
    embedding_pregunta,
    embeddings_documentos
)[0]

print("Dimensiones de la pregunta:")
print(embedding_pregunta.shape)

print("Similitudes encontradas:")

for i in range(len(documentos_permitidos)):
    documento = documentos_permitidos[i]
    print(
        documento["id"],
        round(similitudes[i], 3)
    )

Dimensiones de la pregunta:
(1, 384)
Similitudes encontradas:
ES-VAC-01 0.352
ES-FOR-01 0.447


In [ ]:
#Seleccionar el mejor documento

indice_mejor = similitudes.argmax()
#.argmax() devuelve el índice donde se encuentra el valor más alto, no el valor:
#similitudes = [0.54, 0.30]
#similitudes.argmax()  # devuelve 0

documento_recuperado = documentos_permitidos[indice_mejor]

print("Índice seleccionado:")
print(indice_mejor)

print("Documento recuperado:")
print(documento_recuperado["id"])
print(documento_recuperado["texto"])

print("Puntuación:")
print(round(similitudes[indice_mejor], 3))


Índice seleccionado:
1
Documento recuperado:
ES-FOR-01
Cada empleado dispone de 20 horas anuales de formación.
Puntuación:
0.447


In [ ]:
#Convertir la recuperación en una función

#resumido:

def recuperar_documento(
    pregunta,
    modelo,
    documentos,
    embeddings
):
    embedding_pregunta = modelo.encode([pregunta])

    similitudes = cosine_similarity(
        embedding_pregunta,
        embeddings
    )[0]

    indice_mejor = int(similitudes.argmax())
    documento_recuperado = documentos[indice_mejor]
    puntuacion = float(similitudes[indice_mejor])

    return documento_recuperado, puntuacion



In [ ]:
#Prueba minima:

documento, puntuacion = recuperar_documento(
    pregunta="¿Cuál es mi salario mensual?",
    modelo=modelo_embeddings,
    documentos=documentos_permitidos,
    embeddings=embeddings_documentos
)

print(documento["id"])
print(documento["texto"])
print(round(puntuacion, 3))

assert documento["id"] == "ES-VAC-01"
assert isinstance(puntuacion, float)

print("Prueba superada correctamente.")

ES-VAC-01
Los empleados disponen de 22 días laborables de vacaciones.
0.306
Prueba superada correctamente.


In [ ]:
#Status:

"""
vamos en este punto:

Pregunta
   ↓
Embedding
   ↓
Similitud coseno
   ↓
ES-VAC-01 — 0.542

"""

'\nvamos en este punto:\n\nPregunta\n   ↓\nEmbedding\n   ↓\nSimilitud coseno\n   ↓\nES-VAC-01 — 0.542\n\n'

In [ ]:
#Ahora comienza la A de RAG: Augmented
#Vamos a unir la pregunta con el documento recuperado antes de enviarlos al modelo generativo.

#CONSTRUIR LOS MENSAJES PARA EL LLM

def construir_mensajes(
    pregunta: str,
    documento_recuperado: dict[str, str]
) -> list[dict[str, str]]:
    """
    Construye los mensajes que se enviarán al modelo generativo.

    El documento recibido debe haber pasado previamente por el filtro
    de autorización y por el proceso de recuperación.
    """

    instrucciones = (
        "Eres un asistente de políticas internas. "
        "Responde únicamente con la información incluida en el documento "
        "autorizado. Si el documento no contiene la respuesta, responde: "
        "'No tengo información suficiente en los documentos autorizados'."
    )

    consulta_con_contexto = f"""
DOCUMENTO AUTORIZADO:
ID: {documento_recuperado["id"]}

<contenido>
{documento_recuperado["texto"]}
</contenido>

PREGUNTA DEL USUARIO:
{pregunta}
"""

    mensajes = [
        {
            "role": "system",
            "content": instrucciones
        },
        {
            "role": "user",
            "content": consulta_con_contexto
        }
    ]

    return mensajes

In [ ]:
#Probando la función construir_mensajes

pregunta_usuario = "¿Cuál es mi salario mensual?"

documento_recuperado, puntuacion = recuperar_documento(
    pregunta=pregunta_usuario,
    modelo=modelo_embeddings,
    documentos=documentos_permitidos,
    embeddings=embeddings_documentos
)

mensajes = construir_mensajes(
    pregunta=pregunta_usuario,
    documento_recuperado=documento_recuperado
)

for mensaje in mensajes:
    print("ROL:", mensaje["role"])
    print(mensaje["content"])
    print()

ROL: system
Eres un asistente de políticas internas. Responde únicamente con la información incluida en el documento autorizado. Si el documento no contiene la respuesta, responde: 'No tengo información suficiente en los documentos autorizados'.

ROL: user

DOCUMENTO AUTORIZADO:
ID: ES-VAC-01

<contenido>
Los empleados disponen de 22 días laborables de vacaciones.
</contenido>

PREGUNTA DEL USUARIO:
¿Cuál es mi salario mensual?




In [ ]:
#Quinto bloque: Generacion

from transformers import pipeline

generador = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"
)

salida = generador(
    mensajes,
    max_new_tokens=80,
    do_sample=False
)

respuesta = salida[0]["generated_text"][-1]["content"]


print("Respuesta del asistente:")
print(respuesta)

print("Fuente recuperada:")
print(documento_recuperado["id"])

assert isinstance(respuesta, str)
assert respuesta.strip() != ""

print("Generación completada correctamente.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Respuesta del asistente:
No tengo información suficiente en los documentos autorizados para proporcionar un salario mensual.
Fuente recuperada:
ES-VAC-01
Generación completada correctamente.
